# Advanced CNN
Courtesy of ChatGPT

In [1]:
import torch
from torch import nn
from torch.nn import functional as F

class AdvancedCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )
        self.conv2 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )
        self.conv3 = nn.Sequential(
            nn.Conv2d(64, 128, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )
        self.fc1 = nn.Sequential(
            nn.Linear(128 * 3 * 3, 256), # 128 * 3 * 3 after pooling
            nn.ReLU()
        )
        self.fc2 = nn.Linear(256, 10)
        self.dropout = nn.Dropout(0.5)
    
    def forward(self, x):
        # unflatten inputs into correct shape for model
        x = x.view(x.size(0), 1, 28, 28)
        # apply convolutional layers
        x = self.conv1(x)
        x = self.conv2(x)
        x = self.conv3(x)
        # flatten conv output to pass into fully connected layers
        x = torch.flatten(x, 1)
        # apply fully connected layers with dropout for regularization
        x = self.fc1(x)
        x = self.dropout(x)
        x = self.fc2(x)
        return x

In [2]:
import mnist

BATCH_SIZE = 16
training_loader, test_loader = mnist.get_loaders(BATCH_SIZE)

In [3]:
model = AdvancedCNN()
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [ ]:
import trainer

EPOCHS = 10
STOP_LOSS = 0.01
trainer.train_model(
    model,
    training_loader,
    test_loader,
    loss_fn,
    optimizer,
    print_freq=800,
    epochs=EPOCHS,
    stop_loss=STOP_LOSS,
    device="cuda" if torch.cuda.is_available() else "cpu"
)

In [5]:
# save the model
ts_model = torch.jit.script(model)
torch.jit.save(ts_model, "torchscript-models/advanced_cnn.pt")